# TASK 1 · Iris Flower Classification

## Objective
Train classification models to identify **Setosa, Versicolor, or Virginica** from physical flower measurements.


In [ ]:
iris = load_iris()

iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df["species"] = pd.Categorical.from_codes(
    iris.target, categories=iris.target_names
)

display(iris_df.head())

## 1.1 Exploratory Data Analysis


In [ ]:
print("Dataset shape:", iris_df.shape)
print("\nData types:")
print(iris_df.dtypes)

print("\nMissing values:")
display(iris_df.isnull().sum().to_frame("Missing Values"))

print("\nTotal missing values:", iris_df.isnull().sum().sum())
print("Duplicate rows:", iris_df.duplicated().sum())

print("\nDescriptive statistics:")
display(iris_df.describe().round(2))

print("\nClass distribution:")
display(iris_df["species"].value_counts())

In [ ]:
iris_df.info()

## 1.2 Pairplot — Feature Distributions by Species


In [ ]:
sns.pairplot(
    iris_df,
    hue="species",
    diag_kind="hist",
    height=2.2
)
plt.suptitle("The Iris Feature Relationships by Species", y = 1.02)
plt.show()

## 1.3 The Box Plots for Each Feature


In [ ]:
features = iris.feature_names

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, feature in zip(axes.flatten(), features):
    sns.boxplot(data=iris_df, x="species", y=feature, ax=ax)
    ax.set_title(f"{feature.title()} by Species")
    ax.set_xlabel("Species")
    ax.set_ylabel(feature)

plt.tight_layout()
plt.show()

## 1.4 Feature Selection




In [ ]:
X_iris = iris_df[features]
y_iris = iris_df["species"]

f_scores, p_values = f_classif(X_iris, y_iris)

iris_feature_scores = pd.DataFrame({
    "Feature": features,
    "ANOVA_F_Score": f_scores,
    "p_value": p_values
}).sort_values("ANOVA_F_Score", ascending=False)

display(iris_feature_scores.round(6))
print(
    "Most discriminative features:",
    ", ".join(iris_feature_scores["Feature"].head(2))
)

## 1.5 Train/Test Split of the data


In [ ]:
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris,
    y_iris,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_iris
)

print("Training samples:", len(X_train_iris))
print("Testing samples:", len(X_test_iris))

## 1.6 Training of the Two Classification Models


In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

rf_classifier = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE
)

logistic_model.fit(X_train_iris, y_train_iris)
rf_classifier.fit(X_train_iris, y_train_iris)

logistic_pred = logistic_model.predict(X_test_iris)
rf_pred = rf_classifier.predict(X_test_iris)

print("Both classifiers trained successfully.")

## 1.7 Model Evaluation


In [ ]:
# Evaluation on both models
results = []
models = [
    ("Logistic Regression", logistic_pred),
    ("Random Forest", rf_pred)
]

for name, predictions in models:
    acc = accuracy_score(y_test_iris, predictions)
    results.append({"Model": name, "Accuracy": acc})
    
    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f}\n")
    print(classification_report(y_test_iris, predictions))
    
# Plotting of the confusion matrix
    cm = confusion_matrix(y_test_iris, predictions, labels=iris.target_names)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=iris.target_names, yticklabels=iris.target_names)
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
print("\n" + "="*50)
print(results_df.to_string(index=False))

## 1.8 Best Model


In [ ]:
# Find the best model
best_model = results_df.iloc[0]

print(f"Best model: {best_model['Model']} ({best_model['Accuracy']:.2%} accuracy)")